In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── 1. Load PFR weekly data for 2020-2024 ────────────────────────────────────
all_seasons = []
for season in [2020, 2021, 2022, 2023, 2024]:
    df = pd.read_csv(
        f'https://github.com/nflverse/nflverse-data/releases/download/pfr_advstats/advstats_week_def_{season}.csv',
        low_memory=False
    )
    df['season'] = season
    all_seasons.append(df)
    print(f"Loaded {season}")

pfr_all = pd.concat(all_seasons, ignore_index=True)

# ── 2. Aggregate to season level ─────────────────────────────────────────────
pfr_season = pfr_all[pfr_all['game_type'] == 'REG'].groupby(
    ['season', 'pfr_player_name', 'pfr_player_id']
).agg(
    tgt          = ('def_targets',               'sum'),
    cmp          = ('def_completions_allowed',   'sum'),
    int_         = ('def_ints',                  'sum'),
    td           = ('def_receiving_td_allowed',  'sum'),
    yds          = ('def_yards_allowed',         'sum'),
    avg_rat      = ('def_passer_rating_allowed', 'mean'),
    games_played = ('week',                      'nunique'),
).reset_index()

pfr_season['yds_per_tgt'] = pfr_season['yds'] / pfr_season['tgt']

# ── 3. Load snap counts ──────────────────────────────────────────────────────
snap_all = []
for season in [2020, 2021, 2022, 2023, 2024]:
    df = pd.read_csv(
        f'https://github.com/nflverse/nflverse-data/releases/download/snap_counts/snap_counts_{season}.csv.gz',
        compression='gzip', low_memory=False
    )
    df['season'] = season
    snap_all.append(df)
    print(f"Loaded snaps {season}")

snaps_all = pd.concat(snap_all, ignore_index=True)

cb_snaps_all = (
    snaps_all[snaps_all['position'] == 'CB']
    .groupby(['season', 'pfr_player_id', 'player'])
    .agg(season_snaps=('defense_snaps', 'sum'))
    .reset_index()
)

# ── 4. Merge and compute features ────────────────────────────────────────────
cb_all = pfr_season.merge(
    cb_snaps_all[cb_snaps_all['season_snaps'] >= 300],
    on=['season', 'pfr_player_id'],
    how='inner'
)

cb_all = cb_all[cb_all['tgt'] >= 20].copy()

cb_all['incompletions']     = cb_all['tgt'] - cb_all['cmp'] - cb_all['int_']
cb_all['incompletion_rate'] = cb_all['incompletions'] / cb_all['tgt']
cb_all['int_rate']          = cb_all['int_']          / cb_all['tgt']
cb_all['passer_rating_inv'] = 158.3 - cb_all['avg_rat']
cb_all['target_rate']       = cb_all['tgt']           / cb_all['season_snaps']
cb_all['target_rate_inv']   = 1 - cb_all['target_rate']

print(f"\nTotal CB-seasons: {len(cb_all)}")

# ── 5. PCA ranking function ───────────────────────────────────────────────────
def rank_cbs_pca(season, min_targets=40, min_snaps=600, min_games=13):
    """
    Rank cornerbacks for a given NFL season using PCA on coverage statistics.

    Parameters:
        season     (int): NFL season year (e.g. 2024)
        min_targets (int): Minimum targets to qualify (default 40)
        min_snaps   (int): Minimum defensive snaps to qualify (default 600)
        min_games   (int): Minimum games played to qualify (default 13)

    Returns:
        DataFrame: Ranked CBs with PCA score and all features
    """
    df = cb_all[
        (cb_all['season'] == season) &
        (cb_all['tgt'] >= min_targets) &
        (cb_all['season_snaps'] >= min_snaps) &
        (cb_all['games_played'] >= min_games)
    ].copy()

    if len(df) < 5:
        print(f"Not enough CBs in {season} — try lowering filters")
        return None

    # Winsorize yds_per_tgt at 5th and 95th percentile within season
    p5  = df['yds_per_tgt'].quantile(0.05)
    p95 = df['yds_per_tgt'].quantile(0.95)
    df['yds_per_tgt_wins'] = df['yds_per_tgt'].clip(lower=p5, upper=p95)
    df['yds_per_tgt_inv']  = 1 / (df['yds_per_tgt_wins'] + 1)

    features = ['incompletion_rate', 'int_rate', 'passer_rating_inv',
                'target_rate_inv', 'yds_per_tgt_inv']

    # Normalize within season — era-adaptive
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[features])

    # PCA — first component captures most variance
    pca = PCA(n_components=1)
    scores = pca.fit_transform(X_scaled)
    df['pca_score'] = scores[:, 0]

    # Ensure higher score = better CB
    if df['pca_score'].corr(df['incompletion_rate']) < 0:
        df['pca_score'] = -df['pca_score']

    # Print adaptive weightages
    loadings = pca.components_[0]
    total    = np.abs(loadings).sum()
    print(f"\n{season} adaptive weightages:")
    for feat, loading in zip(features, loadings):
        direction = '+' if loading > 0 else '-'
        print(f"  {feat}: {np.abs(loading)/total*100:.1f}%")
    print(f"  Variance explained: {pca.explained_variance_ratio_[0]*100:.1f}%")
    print(f"  CBs ranked: {len(df)}")

    ranked = df[[
        'player', 'season', 'games_played', 'tgt',
        'incompletion_rate', 'int_rate', 'target_rate',
        'yds_per_tgt', 'avg_rat', 'pca_score'
    ]].sort_values('pca_score', ascending=False).reset_index(drop=True)

    ranked.index += 1
    ranked.insert(0, 'rank', ranked.index)

    return ranked

# ── 6. Run for all seasons and save ─────────────────────────────────────────
all_rankings = []

for season in [2020, 2021, 2022, 2023, 2024]:
    print(f"\n{'='*60}")
    print(f"Season: {season}")
    ranked = rank_cbs_pca(season)
    if ranked is not None:
        all_rankings.append(ranked)
        print(f"\nTop 10:")
        print(ranked.head(10)[['rank', 'player', 'tgt', 'incompletion_rate',
                                'int_rate', 'yds_per_tgt', 'avg_rat', 'pca_score']].to_string())

# Combine all seasons
final = pd.concat(all_rankings, ignore_index=True)
final.to_csv('cb_pca_rankings_2020_2024.csv', index=False)
print(f"\nSaved: cb_pca_rankings_2020_2024.csv")
print(f"Total rows: {len(final)}")
print(f"\nUsage: rank_cbs_pca(2024) — returns full ranked dataframe for that season")

In [ ]:
# ── Load additional seasons ───────────────────────────────────────────────────
extra_seasons = []
for season in [2019, 2025]:
    try:
        df = pd.read_csv(
            f'https://github.com/nflverse/nflverse-data/releases/download/pfr_advstats/advstats_week_def_{season}.csv',
            low_memory=False
        )
        df['season'] = season
        extra_seasons.append(df)
        print(f"Loaded {season}: {len(df)} rows")
    except Exception as e:
        print(f"Failed {season}: {e}")

extra_snaps = []
for season in [2019, 2025]:
    try:
        df = pd.read_csv(
            f'https://github.com/nflverse/nflverse-data/releases/download/snap_counts/snap_counts_{season}.csv.gz',
            compression='gzip', low_memory=False
        )
        df['season'] = season
        extra_snaps.append(df)
        print(f"Loaded snaps {season}")
    except Exception as e:
        print(f"Failed snaps {season}: {e}")

# ── Aggregate new seasons ─────────────────────────────────────────────────────
pfr_extra = pd.concat(extra_seasons, ignore_index=True)

pfr_extra_season = pfr_extra[pfr_extra['game_type'] == 'REG'].groupby(
    ['season', 'pfr_player_name', 'pfr_player_id']
).agg(
    tgt          = ('def_targets',               'sum'),
    cmp          = ('def_completions_allowed',   'sum'),
    int_         = ('def_ints',                  'sum'),
    td           = ('def_receiving_td_allowed',  'sum'),
    yds          = ('def_yards_allowed',         'sum'),
    avg_rat      = ('def_passer_rating_allowed', 'mean'),
    games_played = ('week',                      'nunique'),
).reset_index()

pfr_extra_season['yds_per_tgt'] = pfr_extra_season['yds'] / pfr_extra_season['tgt']

# ── Snap counts for new seasons ───────────────────────────────────────────────
snaps_extra = pd.concat(extra_snaps, ignore_index=True)

cb_snaps_extra = (
    snaps_extra[snaps_extra['position'] == 'CB']
    .groupby(['season', 'pfr_player_id', 'player'])
    .agg(season_snaps=('defense_snaps', 'sum'))
    .reset_index()
)

# ── Merge and compute features ────────────────────────────────────────────────
cb_extra = pfr_extra_season.merge(
    cb_snaps_extra[cb_snaps_extra['season_snaps'] >= 300],
    on=['season', 'pfr_player_id'],
    how='inner'
)

cb_extra = cb_extra[cb_extra['tgt'] >= 20].copy()

cb_extra['incompletions']     = cb_extra['tgt'] - cb_extra['cmp'] - cb_extra['int_']
cb_extra['incompletion_rate'] = cb_extra['incompletions'] / cb_extra['tgt']
cb_extra['int_rate']          = cb_extra['int_']          / cb_extra['tgt']
cb_extra['passer_rating_inv'] = 158.3 - cb_extra['avg_rat']
cb_extra['target_rate']       = cb_extra['tgt']           / cb_extra['season_snaps']
cb_extra['target_rate_inv']   = 1 - cb_extra['target_rate']

# ── Append to main dataset ────────────────────────────────────────────────────
cb_all = pd.concat([cb_all, cb_extra], ignore_index=True)

print(f"\nUpdated CB-seasons: {len(cb_all)}")
print(f"Seasons available: {sorted(cb_all['season'].unique())}")

# ── Test on 2019 and 2025 ─────────────────────────────────────────────────────
for season in [2019, 2025]:
    print(f"\n{'='*60}")
    print(f"Season: {season}")
    ranked = rank_cbs_pca(season)
    if ranked is not None:
        print(f"\nTop 10:")
        print(ranked.head(10)[['rank', 'player', 'tgt', 'incompletion_rate',
                                'int_rate', 'yds_per_tgt', 'avg_rat', 'pca_score']].to_string())